In [ ]:
import sys
print(sys.executable)

import torch
print(torch.__version__)

In [ ]:
%pip install sentence-transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import pipeline

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded successfully!")

In [ ]:
documents = [
    "Students can apply for scholarships and financial aid.",
    "Python is a popular programming language used in AI.",
    "The university library is open from 9AM to 8PM.",
    "Students can complete their internship during their studies.",
    "Machine learning allows computers to learn from data.",
    "The university offers courses in Python programming, machine learning, and artificial intelligence.",
    "Students must complete all required examinations and credits to graduate.",
    "The university provides career counseling and internship guidance for students."
]
# query = "How can i get financial support for my education.?"
query = "Is Python a Programming Language?"
print("User Question: ",query)

In [ ]:
document_embeddings = model.encode(documents)
print("Document Embeddings Shape: ",document_embeddings.shape)

In [ ]:
query_embedding = model.encode([query])
similarity_score = cosine_similarity(
    query_embedding,
    document_embeddings
)
print("Cosine Similarity Score: ", similarity_score)

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

print("Loading language model...")
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
generator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
def retrieve_documents(query, top_k=2):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, document_embeddings)[0]

    ranked_indices = np.argsort(similarities)[::-1]
    top_indices = ranked_indices[:top_k]
    retrieved_documents = []

    for index in top_indices:
        retrieved_documents.append({
            "text": documents[index],
            "score": float(similarities[index])
        })

    return retrieved_documents

In [ ]:
# Call the retrieve_documents function defined in the previous cell
retrieved_results = retrieve_documents(query, top_k=2)
print(retrieved_results)

In [ ]:
def create_prompt(query, retrieved_results):
    context="\n".join(
        item["text"]
        for item in retrieved_results
    )

    prompt=f"""
    answer the following question using context below.
    if the context doesn't have the answer , say:
      i don't have enough info to answer.

    context:
    {context}

    question:
    {query}

    answer:
    """

    return prompt

In [ ]:
def rag_chatbot(query):
  retrieved_docs = retrieve_documents(query, top_k=2)
  prompt = create_prompt(query, retrieved_docs)
  inputs = tokenizer(prompt, return_tensors="pt")
  outputs = generator.generate(**inputs, max_new_tokens=150)
  answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
  print("answer:", answer)

In [ ]:
retrieved_docs = retrieve_documents(
    query,
    top_k=2
)

print("\nretrieved docs:")

for i, item in enumerate(retrieved_docs):
  print(f"{i+1}. {item['text']}")

  print(f"similarity_scores:{item['score']}")

In [ ]:
prompt=create_prompt(
    query,
    retrieved_docs
)
print(prompt)


In [ ]:

inputs = tokenizer(prompt, return_tensors="pt")


outputs = generator.generate(
    **inputs,
    max_new_tokens=150
)


result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

In [ ]:
while True:
  user_questions = input("question: ")
  if len(user_questions.strip()) == 0:
    print("please enter a valid question")
    continue

  if user_questions.strip().lower() == "exit":
    break

  rag_chatbot(user_questions)